In [44]:
import torch
from torch import nn
import torchvision
from torchvision import transforms
print(torch.__version__)
print(torchvision.__version__)

2.10.0+cu128
0.25.0+cu128


In [43]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
Tesla T4


In [42]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## getting dataset

In [ ]:
from torchvision import datasets, transforms


train_data = datasets.FashionMNIST(
    root="../cnn/data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
    target_transform=None
)

test_data = datasets.FashionMNIST(
    root="../cnn/data",
    train=False,
    download=True,
    transform=transforms.ToTensor(),
    target_transform=None
)

In [ ]:
img,label = train_data[0]
print(img.shape)
print(label)

In [ ]:
class_name = train_data.classes
cls_ind = train_data.class_to_idx
print(class_name)
print(cls_ind)
print(train_data.targets)

## visualize the sample data

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(img.squeeze(),cmap="gray")
plt.axis(False)

In [ ]:
torch.manual_seed(42)
fig = plt.figure(figsize=(9,9))
rows,cols = 4,4
for i in range(1,rows*cols+1):
    rand_ind = torch.randint(0,len(train_data),size=[1]).item()
    print(rand_ind)
    image,labels = train_data[rand_ind]
    fig.add_subplot(rows,cols,i)
    plt.imshow(image.squeeze(),cmap="gray")
    plt.title(class_name[labels])
    plt.axis(False)


## dataloader and minibatches

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32
train_data_loader = DataLoader(dataset = train_data,
                               batch_size = BATCH_SIZE,
                               shuffle = True)

test_data_loader = DataLoader(dataset = test_data,
                              batch_size = BATCH_SIZE,
                              shuffle =False)

print(train_data_loader)
print(test_data_loader)
train_feat,train_label = next(iter(train_data_loader))
torch.manual_seed(42)
ran_ind = torch.randint(0,len(train_feat),size=[1]).item()
image1,label1 =  train_feat[ran_ind],train_label[ran_ind]
plt.imshow(image1.squeeze(),cmap="gray")
plt.title(class_name[label1])
plt.axis(False)
print(image1.shape)

## model

In [ ]:
class fashionMNISTmodel(nn.Module):
    def __init__(self,
                 input_size:int,
                 hidden_units:int,
                 output_size:int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_size,out_features=hidden_units),
            nn.Linear(in_features=hidden_units,out_features=output_size)
        )

    def forward(self,x):
        return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)
model = fashionMNISTmodel(input_size=784,
                          hidden_units=10,
                          output_size=len(class_name))

print(model)

In [ ]:
dummy_x = torch.rand([1,1,28,28])
print(model(dummy_x))
print(model.state_dict())

## loss function and optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(),lr=0.1)

## tracking model performance and speed

In [ ]:
from timeit import default_timer as timer

def print_train_time(start:float,
                     end:float,
                     device:torch.device=None):
    total_time= end - start
    print(f"total time:{total_time:.3f},device name :{device}")
    return total_time

In [ ]:
start =timer()
print("hi")
end =timer()
print(print_train_time(start=start,end=end,device="cpu"))

## create training loop and train model on batches

In [ ]:
from tqdm.auto import tqdm

def accuracy_func(y_true, y_pred):
    correct = torch.eq(y_true, y_pred.argmax(dim=1)).sum().item()
    acc = (correct / len(y_true)) * 100
    return acc

torch.manual_seed(42)
train_start = timer()

epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"epoch:{epoch}\n----")
    train_loss = 0
    train_acc = 0

    for batch, (x, y) in enumerate(train_data_loader):
        model.train()
        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        train_loss += loss
        train_acc += accuracy_func(y, y_pred)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 400 == 0:
            print(f"looked at {batch * len(x)}/{len(train_data_loader.dataset)}")

    train_loss /= len(train_data_loader)
    train_acc /= len(train_data_loader)

    test_loss, test_acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for x_test, y_test in test_data_loader:
            test_pred = model(x_test)
            test_loss += loss_fn(test_pred, y_test)
            test_acc += accuracy_func(y_test, test_pred)

        test_loss /= len(test_data_loader)
        test_acc /= len(test_data_loader)

    print(f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.2f}%")
    print(f"test_loss: {test_loss:.4f} | test_acc: {test_acc:.2f}%")
end_train= timer()
train_model_time = print_train_time(start=train_start,end=end_train,device=str(next(model.parameters()).device))

In [ ]:
torch.manual_seed(42)
def eval_model(model:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               accuracy_func,
               device:torch.device):
    loss,acc =0,0
    model.eval()
    with torch.inference_mode():
        for x,y in data_loader:
            x1,y1 = x.to(device),y.to(device)
            y_pred1 = model(x1)
            loss+=loss_fn(y_pred1,y1)
            acc+= accuracy_func(y1, y_pred1)

        loss/= len(data_loader)
        acc/=len(data_loader)

    return {"model_name":model.__class__.__name__,
            "model_loss":loss.item(),
            "model_acc":acc}      

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model_results = eval_model(model=model,
                           data_loader=test_data_loader,
                           loss_fn=loss_fn,
                           accuracy_func=accuracy_func,
                           device=device)

print(model_results)

In [ ]:
class fashionMNISTmodel1(nn.Module):
    def __init__(self,
                 input_size:int,
                 hidden_units:int,
                 output_size:int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_size,out_features=hidden_units),
            nn.LeakyReLU(),
            nn.Linear(in_features=hidden_units,out_features=output_size),
            nn.LeakyReLU()
        )

    def forward(self,x):
        return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)
model1 = fashionMNISTmodel1(input_size=784,
                          hidden_units=10,
                          output_size=len(class_name)).to(device=device)

print(next(model.parameters()).device)

## loss func and optimizer


In [ ]:
loss_fn1 = nn.CrossEntropyLoss()
optimizer1 = torch.optim.SGD(params=model1.parameters(),
                             lr=0.1)

In [33]:
def train_step(model1:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_func:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               accuracy_fn,
               device:torch.device = device):


    train_loss1 = 0
    train_acc1 = 0
    model1.train()

    for batch1, (x, y) in enumerate(data_loader):
        x2,y2 = x.to(device),y.to(device)
        y_pred2 = model1(x2)
        loss = loss_func(y_pred2, y2)
        train_loss1 += loss
        train_acc1 += accuracy_fn(y2, y_pred2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        
    train_loss1 /= len(data_loader)
    train_acc1 /= len(data_loader)
    print(f"train_loss: {train_loss1:.4f} | train_acc: {train_acc1:.2f}%")

In [ ]:
def test_step(model1:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_func:torch.nn.Module,
               accuracy_fn,
               device:torch.device = device):

    test_loss1, test_acc1 = 0, 0
    model1.eval()
    with torch.inference_mode():
        for x_test1, y_test1 in data_loader:
            x_test1, y_test1=x_test1.to(device), y_test1.to(device)
            test_pred1 = model1(x_test1)
            test_loss1 += loss_func(test_pred1, y_test1)
            test_acc1 += accuracy_fn(y_test1, test_pred1)

        test_loss1 /= len(data_loader)
        test_acc1 /= len(data_loader)

    print(f"test_loss: {test_loss1:.4f} | test_acc: {test_acc1:.2f}%")


In [ ]:
torch.manual_seed(42)
from timeit import default_timer as timer
train_start_time = timer()
epochs =3
for epoch in tqdm(range(epochs)):
    print(f"epoch:{epoch}\n----")
    train_step(model1 = model1,
               data_loader=train_data_loader,
               loss_func=loss_fn1,
               optimizer=optimizer1,
               accuracy_fn=accuracy_func,
               device=device)

    test_step(model1=model1,
              data_loader=test_data_loader,
              loss_func=loss_fn1,
              accuracy_fn=accuracy_func,
              device=device)
    
end_train_time = timer()
total_time = print_train_time(start=train_start_time,end=end_train_time,
                              device=device)   

In [ ]:
torch.manual_seed(42)
def eval_model(model:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               accuracy_func,
               device:torch.device):
    loss1,acc1 =0,0
    model.eval()
    with torch.inference_mode():
        for x,y in data_loader:
            xa,ya = x.to(device),y.to(device)
            y_preda = model(xa)
            loss1+=loss_fn(y_preda,ya)
            acc1+= accuracy_func(ya, y_preda)

        loss1/= len(data_loader)
        acc1/=len(data_loader)

    return {"model_name":model.__class__.__name__,
            "model_loss":loss1.item(),
            "model_acc":acc1}      

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1.to(device)

model_results = eval_model(model=model,
                           data_loader=test_data_loader,
                           loss_fn=loss_fn,
                           accuracy_func=accuracy_func,
                           device=device)

print(model_results)

In [ ]:
print(img.shape)

In [61]:
class model(nn.Module):
    def __init__(self,input_unit:int,hidden_units:int,output_unit:int):
        super().__init__()
        self.cov_layer1 = nn.Sequential(
            nn.Conv2d(in_channels=input_unit,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)                    
        )

        self.cov_layer2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)                    
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*7*7,
                      out_features=output_unit)
        )

    def forward(self,x):
        x = self.cov_layer1(x)
        print(x.shape)
        x = self.cov_layer2(x)
        print(x.shape)
        x = self.classifier(x)
        return x      

In [30]:
torch.manual_seed(42)
Model11 = model(input_unit=1,
              hidden_units=10,
              output_unit=len(class_name)).to(device)

In [ ]:
rand_img = torch.randn(size= (1,28,28))
print(rand_img.shape)

In [31]:
Model11(rand_img.unsqueeze(0).to(device))

torch.Size([1, 10, 14, 14])
torch.Size([1, 10, 7, 7])


tensor([[ 0.0366, -0.0940,  0.0686, -0.0485,  0.0068,  0.0290,  0.0132,  0.0084,
         -0.0030, -0.0185]], device='cuda:0', grad_fn=<AddmmBackward0>)

## setup loss fun and optimizer

In [62]:
loss_function = nn.CrossEntropyLoss()
optimizer_adam =torch.optim.SGD(params=Model11.parameters(),lr=0.001)

In [63]:
print(Model11.state_dict())

OrderedDict({'cov_layer1.0.weight': tensor([[[[-0.4011, -0.3792, -0.7340],
          [-0.3497, -0.7289, -0.5886],
          [-0.8182, -0.4601, -0.3620]]],


        [[[-0.7023, -0.1683, -0.3958],
          [-0.2116, -0.4132, -0.2978],
          [-0.5051, -0.2014, -0.4092]]],


        [[[-0.8115, -0.3711, -0.6099],
          [-0.4950, -0.5914, -0.2353],
          [-0.7190, -0.6097, -0.5505]]],


        [[[-0.6557, -0.4233, -0.7837],
          [-0.1544, -0.7380, -0.1973],
          [-0.3998, -0.5629, -0.2485]]],


        [[[-0.5485, -0.3835, -0.5764],
          [-0.7074, -0.5828, -0.7289],
          [-0.5120, -0.3632, -0.4607]]],


        [[[ 0.5103,  0.8461,  0.7133],
          [ 0.8255,  0.4520,  0.3241],
          [ 0.5286,  0.4005,  0.9287]]],


        [[[-0.3590, -0.3163, -0.3485],
          [-0.4608, -0.1935, -0.6907],
          [-0.4340, -0.6818, -0.3511]]],


        [[[-0.5906, -0.3681, -0.5398],
          [-0.1981, -0.6704, -0.6719],
          [-0.6729, -0.1696, -0.3591]]]

In [64]:
def accuracy_fn1(y_true, y_pred):
    y_pred_class = y_pred.argmax(dim=1)
    return (y_pred_class == y_true).sum().item() / len(y_true) * 100


In [65]:
def train_step1(model11:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_func:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               accuracy_fn1,
               device:torch.device = device):


    train_loss1 = 0
    train_acc1 = 0
    model11.train()

    for batch1, (x, y) in enumerate(data_loader):
        x2,y2 = x.to(device),y.to(device)
        y_pred2 = model11(x2)
        loss = loss_func(y_pred2, y2)
        train_loss1 += loss
        train_acc1 += accuracy_fn1(y2, y_pred2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        
    train_loss1 /= len(data_loader)
    train_acc1 /= len(data_loader)
    print(f"train_loss: {train_loss1:.4f} | train_acc: {train_acc1:.2f}%")

In [66]:
def test_step1(model11:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_func:torch.nn.Module,
               accuracy_fn1,
               device:torch.device = device):

    test_loss1, test_acc1 = 0, 0
    model11.eval()
    with torch.inference_mode():
        for x_test1, y_test1 in data_loader:
            x_test1, y_test1=x_test1.to(device), y_test1.to(device)
            test_pred1 = model11(x_test1)
            test_loss1 += loss_func(test_pred1, y_test1)
            test_acc1 += accuracy_fn1(y_test1, test_pred1)
            test_loss11 = test_loss1

        test_loss1 /= len(data_loader)
        test_acc1 /= len(data_loader)

    print(f"test_loss: {test_loss11:.4f} | test_acc: {test_acc1:.2f}%")


## training and testing loop


In [67]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

from timeit import default_timer as Timer
start_train=Timer()
epochs =3 
for epoch in tqdm(range(epochs)):
    print(f"epoch:{epoch}\n-----")
    train_step1(model11=Model11,
               data_loader=train_data_loader,
               loss_func=loss_function,
               optimizer=optimizer_adam,
               accuracy_fn1=accuracy_fn1,
               device=device
               )
    test_step1(model11=Model11,
              data_loader=test_data_loader,
              loss_func=loss_function,
              accuracy_fn1=accuracy_fn1,
              device=device)
    
train_model_time_s =Timer() 
total = print_train_time(start_train,train_model_time_s,device=device)    

  0%|          | 0/3 [00:00<?, ?it/s]

epoch:0
-----
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch.Size([32, 10, 7, 7])
torch.Size([32, 10, 14, 14])
torch